# Plane controls, readable addresses, and ROI video

This is a source-preserving UI/export build. It reads the committed full M07 viewer at `7f3bc5e` and writes the revised v3.1 HTML. Geometry is unchanged. Run notebook 19 afterward to regenerate Lite from this full viewer. A full Git clone is required because the source is pinned to a prior commit.


In [1]:
from pathlib import Path
import subprocess, re, json

ROOT = Path.cwd()
BASE_COMMIT = '7f3bc5e'
source = subprocess.run(['git','show',f'{BASE_COMMIT}:viewer/mouse_lung_v3.1.html'],check=True,capture_output=True).stdout.decode('utf-8')
full = source

def swap(old, new):
    global full
    count = full.count(old)
    assert count == 1, (old[:90], count)
    full = full.replace(old, new, 1)



## Organize plane controls

Open View & layers by default, keep Section plane collapsed until Clip is selected, and expose only two drag pads with numeric steppers. The original sliders remain hidden as internal state to preserve existing plane-control behavior. The M07 address is shortened but still includes model fingerprint, coordinate frame, angles and distance. Legacy LP1 and earlier readable addresses remain importable.


In [2]:
# Place layers first and closed-plane controls second. Clip opens the controls.
swap("const views=document.createElement('details');views.className='miniDetails';views.innerHTML=", "const views=document.createElement('details');views.className='miniDetails';views.open=true;views.innerHTML=")
swap("const cuts=document.createElement('details');cuts.className='miniDetails';cuts.open=true;cuts.innerHTML=", "const cuts=document.createElement('details');cuts.className='miniDetails';cuts.id='planeDetails';cuts.open=false;cuts.innerHTML=")
swap('ui.appendChild(cuts);ui.appendChild(views);ui.appendChild(address);', 'ui.appendChild(views);ui.appendChild(cuts);ui.appendChild(address);')
swap("if(clipping){document.getElementById('sectionEvidence').hidden=true;", "if(clipping){document.getElementById('planeDetails').open=true;document.getElementById('sectionEvidence').hidden=true;")

# Keep the existing range elements as JS state, but expose only pads and numeric steppers.
control_js = r"""
 const planeControls=cutBody.querySelector('.planeControls');
 planeControls.querySelector('.planeControlHead strong').textContent='Position & orientation';
 planeControls.querySelector('.planeControlHead span').textContent='M07 local · degrees / mm';
 const tiltGroup=planeControls.querySelector('.tiltPadWrap'),slideGroup=planeControls.querySelector('.offsetPadWrap');
 tiltGroup.querySelector('span:first-child').innerHTML='<b>Tilt plane</b><small>Azimuth ↔ · elevation ↕</small>';
 slideGroup.querySelector('span:first-child').innerHTML='<b>Slide plane</b><small>Normal offset</small>';
 tiltGroup.querySelector('.padHint').remove();
 const tiltInputs=document.createElement('div');tiltInputs.className='padNumbers';
 function addPlaneStepper(target,title,unit,step,parent){
   const input=document.getElementById(target),row=document.createElement('div');row.className='padNumberRow';
   const label=document.createElement('label');label.htmlFor=target;label.textContent=title;
   const shell=document.createElement('div');shell.className='padNumberShell';
   shell.appendChild(input);
   const arrows=document.createElement('div');arrows.className='padStepArrows';
   for(const [glyph,dir] of [['▴',1],['▾',-1]]){
     const button=document.createElement('button');button.type='button';button.textContent=glyph;
     button.setAttribute('aria-label',(dir>0?'Increase ':'Decrease ')+title.toLowerCase());
     button.addEventListener('click',()=>{const next=Math.min(Number(input.max),Math.max(Number(input.min),Number(input.value)+dir*step));input.value=next.toFixed(target==='depthNumber'?2:1);input.dispatchEvent(new Event('input',{bubbles:true}));});
     arrows.appendChild(button);
   }
   shell.appendChild(arrows);const suffix=document.createElement('span');suffix.className='padUnit';suffix.textContent=unit;
   row.append(label,shell,suffix);parent.appendChild(row);
 }
 addPlaneStepper('angleNumber','Azimuth','°',1,tiltInputs);
 addPlaneStepper('angle2Number','Elevation','°',1,tiltInputs);
 tiltGroup.appendChild(tiltInputs);
 const offsetInputs=document.createElement('div');offsetInputs.className='padNumbers';
 addPlaneStepper('depthNumber','Offset','mm',.05,offsetInputs);
 slideGroup.appendChild(offsetInputs);
 slideGroup.querySelector('#offsetPadValue').remove();
"""
swap(" cutBody.appendChild(ui.querySelector('.planeControls'));", " cutBody.appendChild(ui.querySelector('.planeControls'));"+control_js)

style = r"""
/* v3.1: concise pad controls; native range inputs remain as invisible state for the viewer. */
#ui .planeControlRow{display:none}
#ui .planeControls{padding:10px 12px 12px}
#ui .planeControlHead{margin-bottom:9px}
#ui .tiltPadWrap,#ui .offsetPadWrap{display:grid;grid-template-columns:82px 78px minmax(105px,1fr);align-items:center;gap:9px;margin:0;padding:10px 0;color:#c4d6df;border-top:1px solid rgba(145,218,237,.12);font-size:10px}
#ui .tiltPadWrap>span:first-child,#ui .offsetPadWrap>span:first-child{min-width:0}
#ui .tiltPadWrap b,#ui .offsetPadWrap b{display:block;color:#e5f5fa;font-size:10.5px}
#ui .tiltPadWrap small,#ui .offsetPadWrap small{display:block;color:#91aab7;line-height:1.3;margin-top:3px}
#ui #tiltPad{width:76px;height:76px}
#ui #offsetPad{width:78px;min-width:78px;height:53px}
#ui #offsetPad::before{left:4px;top:3px;font-size:8px}#ui #offsetPad::after{right:4px;top:3px;font-size:8px}
#ui .padNumbers{display:grid;gap:5px;min-width:0}
#ui .padNumberRow{display:grid;grid-template-columns:minmax(0,1fr) 75px 15px;gap:4px;align-items:center;font-size:9px}
#ui .padNumberRow label{overflow:hidden;text-overflow:ellipsis;color:#afcbd8}
#ui .padNumberShell{display:flex;border:1px solid #3d697d;border-radius:8px;overflow:hidden;background:#10212d;height:30px}
#ui .padNumberShell input{width:100%;min-width:0;text-align:right;padding:0 3px;border:0;background:transparent;color:#e9f9ff;font:600 11px ui-monospace,SFMono-Regular,monospace;appearance:textfield;-moz-appearance:textfield}
#ui .padNumberShell input::-webkit-inner-spin-button,#ui .padNumberShell input::-webkit-outer-spin-button{-webkit-appearance:none;margin:0}
#ui .padStepArrows{display:flex;flex-direction:column;border-left:1px solid #31566b}
#ui .padStepArrows button{display:block;width:18px;height:14px;min-height:14px;padding:0;border:0;border-radius:0;background:transparent;color:#a9d7e8;font-size:10px;line-height:12px}
#ui .padStepArrows button:hover{background:#224658;color:white}
#ui .padUnit{color:#a7bdc9}
#ui .padNumberShell:focus-within,#ui .padStepArrows button:focus-visible{outline:2px solid #81d7f7;outline-offset:1px}
#ui #planeAddress,#ui #planePaste{font:11px/1.45 ui-monospace,SFMono-Regular,monospace;overflow-wrap:anywhere;background:#0a1821;border:1px solid #477c92;color:#cdf1fb;border-radius:9px;padding:8px}
#ui .planeAddressHelp{font-size:10px;line-height:1.5;color:#acc4d0;margin:0 0 9px}
#ui .planeAddressHelp b{color:#daf2fa}
/* Spacious controls at desktop widths, with one compact mobile breakpoint. */
#ui{width:clamp(350px,24vw,430px)!important;padding:16px!important}
#ui h1{font-size:20px!important}#ui .sub{font-size:12px!important;line-height:1.5!important}
#ui button{min-height:38px;font-size:12.5px!important}#ui .miniDetails summary{font-size:13px;padding:11px 12px}
#ui .planeCard .planeName{font-size:15px}#ui .planeCard .planeLine{font-size:12px}
#ui .tiltPadWrap,#ui .offsetPadWrap{grid-template-columns:95px 100px minmax(120px,1fr);gap:11px;font-size:11px}
#ui #tiltPad{width:92px;height:92px}#ui #offsetPad{width:96px;min-width:96px;height:60px}
#ui .padNumberRow{grid-template-columns:minmax(0,1fr) 88px 18px;font-size:10px}
#ui .padNumberShell{height:36px}#ui .padNumberShell input{font-size:12px}
#ui .padStepArrows button{width:20px;height:17px;min-height:17px;font-size:11px!important}
#presentationToolbar{padding:8px;gap:8px}#presentationToolbar button{min-width:44px;width:44px;height:44px;min-height:44px}#presentationToolbar svg{width:23px;height:23px}
#slicePanel{width:min(440px,calc(100vw - 460px))!important;padding:14px!important}#slicePanel h2{font-size:17px!important}#slicePanel .small{font-size:11px!important}
@media(max-width:760px){#ui{width:calc(100vw - 20px)!important;max-height:58vh!important;padding:12px!important}#ui .tiltPadWrap,#ui .offsetPadWrap{grid-template-columns:78px 80px minmax(92px,1fr);gap:6px}#ui #tiltPad{width:78px;height:78px}#ui #offsetPad{width:78px;min-width:78px;height:54px}#ui .padNumberRow{grid-template-columns:minmax(0,1fr) 74px 14px;gap:3px}#ui .padNumberShell{height:32px}#ui .padStepArrows button{height:15px;min-height:15px}#slicePanel{top:auto!important;bottom:62px!important;right:10px!important;width:calc(100vw - 20px)!important;max-height:34vh!important}#presentationToolbar{top:auto;bottom:10px;right:10px;padding:5px;gap:3px}#presentationToolbar button{min-width:38px;width:38px;height:38px;min-height:38px}}
"""
swap('</style>',style+'</style>')

# Keep a compact, human-readable, typeable address; older six-decimal and LP1 links still parse.
swap('return `M07-RVCR1@${PLANE_MODEL.slice(0,12)};az=${a.toFixed(6)}deg;el=${b.toFixed(6)}deg;d=${d.toFixed(6)}mm`;', 'return `M07-RVCR1@${PLANE_MODEL.slice(0,12)};az=${Number(a.toFixed(2))}deg;el=${Number(b.toFixed(2))}deg;d=${Number(d.toFixed(2))}mm`;')
swap("address.querySelector('.small').textContent='M07 local model frame (mm): right, ventral, cranial per NAV terminology discussed by Ruberte et al. (2025). az and el are plane angles; d is signed distance from the M07 origin. These are not Xenium coordinates.';", "address.querySelector('.small').outerHTML='<p class=\"planeAddressHelp\"><b>How to read this ID:</b> M07 is the reference model; RVCR1 means right–ventral–cranial axes. The 12-character code identifies this mesh. <b>az</b> rotates the plane toward the ventral axis, <b>el</b> tilts it toward cranial, and <b>d</b> slides it along its normal in millimeters. Copy the ID, or type the two lines printed on an export below. This is not a Xenium slide location.</p>';")
swap("document.getElementById('planePaste').placeholder='M07-RVCR1@…;az=…deg;el=…deg;d=…mm';", "document.getElementById('planePaste').placeholder='Type: M07-RVCR1@model;az=15deg;el=0deg;d=-5.95mm';")



## Simplify the export canvas

Remove the blue 2D box and show a typeable plane address on the black canvas. Credit is compact and points to the project repository. This shared compositor feeds PNG, GIF, and supplementary video exports.


In [3]:
# Publication frames: black background, no decorative blue box, readable plane address.
old_start=full.index('    if(forGif&&showSection){')
old_end=full.index('    }else{',old_start)+5
old=full[old_start:old_end]
new=r"""    if(forGif&&showSection){
      const gap=Math.max(12,Math.round(w*.015)),panelX=Math.round(w*.63),leftW=panelX-gap,boxW=w-panelX-gap;
      const cropRatio=leftW/h;let cropH=source.height,cropW=cropH*cropRatio;
      if(cropW>source.width){cropW=source.width;cropH=cropW/cropRatio}
      ctx.drawImage(source,(source.width-cropW)/2,(source.height-cropH)/2,cropW,cropH,0,0,leftW,h);
      ctx.strokeStyle='rgba(160,204,220,.25)';ctx.lineWidth=1;ctx.beginPath();ctx.moveTo(panelX-gap/2,gap);ctx.lineTo(panelX-gap/2,h-gap);ctx.stroke();
      const pad=Math.max(10,Math.round(w*.014)),textX=panelX+pad,innerW=boxW-2*pad;
      ctx.fillStyle='#d9f4fd';ctx.font='700 '+Math.max(14,Math.round(w/95))+'px system-ui,sans-serif';ctx.fillText('M07 MODEL SECTION',textX,gap+pad+3,innerW);
      ctx.fillStyle='#bfd4dd';ctx.font=Math.max(12,Math.round(w/112))+'px system-ui,sans-serif';ctx.fillText(document.getElementById('planeName').textContent,textX,gap+pad+25,innerW);
      const addressY=h-gap-Math.max(105,Math.round(h*.19));
      const section=document.getElementById('sliceCanvas'),imageTop=gap+pad+36,imageH=Math.max(30,Math.min(innerW*section.height/section.width,addressY-imageTop-15)),imageW=Math.min(innerW,imageH*section.width/section.height);
      ctx.drawImage(section,textX+(innerW-imageW)/2,imageTop+(addressY-12-imageTop-imageH)/2,imageW,imageH);
      ctx.strokeStyle='rgba(160,204,220,.3)';ctx.beginPath();ctx.moveTo(textX,addressY-13);ctx.lineTo(textX+innerW,addressY-13);ctx.stroke();
      ctx.fillStyle='#b5e9f5';ctx.font='700 '+Math.max(11,Math.round(w/113))+'px system-ui,sans-serif';ctx.fillText('REOPEN THIS PLANE',textX,addressY+4,innerW);
      const planeID=encodePlane(sectionAngle,sectionTilt,sectionDepth),breakAt=planeID.indexOf(';');
      ctx.fillStyle='#dcecf2';ctx.font=Math.max(10,Math.round(w/120))+'px ui-monospace,SFMono-Regular,monospace';
      ctx.fillText(planeID.slice(0,breakAt+1),textX,addressY+25,innerW);
      ctx.fillText(planeID.slice(breakAt+1),textX,addressY+46,innerW);
      ctx.fillStyle='#9cb6c4';ctx.font=Math.max(10,Math.round(w/130))+'px system-ui,sans-serif';
      ctx.fillText('az / el: tilt in degrees · d: offset in mm',textX,addressY+66,innerW);
      drawCompactCompass(ctx,leftW,h,true);
    }"""
assert old.startswith('    if(forGif&&showSection){') and old.endswith('    }')
full=full[:old_start]+new+full[old_end:]
swap("const line='M07 · az '+Number(angle.value).toFixed(0)+'° · el '+Number(angle2.value).toFixed(0)+'° · d '+Number(depth.value).toFixed(2)+' mm';", "const line=encodePlane(sectionAngle,sectionTilt,sectionDepth);")
swap("const credit='Arash Bagherabadi (2026) · github.com/arashabadi'", "const credit='Created by Arash · github.com/arashabadi/mouse-lung-explorer'")



## Measured E02 ROI exports

The independent E02 tissue-surface ROI remains distinct from M07. Its PNG and video show the 3D surface, acquisition-axis slice, typeable ROI address, the original paper DOI, homepage, and project credit. The browser encodes MP4/H.264 where supported, otherwise WebM. Record and stop use the same button.


In [4]:
# Shared ROI frame for PNG and video, with source paper and slice ID.
roi_video=r"""
  const roiRecordButton=document.getElementById('recordRoiVideo');
  const roiVideoTypes=['video/mp4;codecs=avc1.42E01E','video/mp4;codecs=avc1','video/webm;codecs=vp9','video/webm;codecs=vp8'];
  const roiVideoMime=typeof MediaRecorder==='undefined'?null:roiVideoTypes.find(type=>MediaRecorder.isTypeSupported(type));
  let roiRecorder=null,roiStream=null,roiChunks=[],roiRecording=false,roiVideoRaf=0,roiLastFrame=-Infinity;
  function roiCaptureFrame(width){
    const height=Math.round(width*9/16),out=document.createElement('canvas');out.width=width;out.height=height;
    const ctx=out.getContext('2d');ctx.fillStyle='#05080d';ctx.fillRect(0,0,width,height);
    const pad=Math.max(16,Math.round(width*.025)),top=Math.round(height*.13),bottom=Math.round(height*.21),contentH=height-top-bottom,leftW=Math.round(width*.62),rightX=leftW+pad;
    if(alvRenderer){alvRenderer.render(alvScene,alvCamera);const source=alvRenderer.domElement;ctx.drawImage(source,0,0,source.width,source.height,pad,top,leftW-pad*2,contentH)}
    const slice=document.getElementById('roiSlice'),rightW=width-rightX-pad;
    if(slice.width&&slice.height){const scale=Math.min(rightW/slice.width,contentH/slice.height),dw=slice.width*scale,dh=slice.height*scale;ctx.imageSmoothingEnabled=false;ctx.drawImage(slice,0,0,slice.width,slice.height,rightX+(rightW-dw)/2,top+(contentH-dh)/2,dw,dh);ctx.imageSmoothingEnabled=true}
    ctx.fillStyle='#ddf2f8';ctx.font='700 '+Math.round(width*.025)+'px system-ui,sans-serif';ctx.fillText('CHICKEN-WIRE VIEW · E02 ROI',pad,Math.round(height*.055));
    ctx.fillStyle='#b6cbd4';ctx.font=Math.round(width*.014)+'px system-ui,sans-serif';ctx.fillText('Exploratory tissue-surface reconstruction · independent specimen',pad,Math.round(height*.09));
    const y=Math.round(height*.82);ctx.strokeStyle='#3d6474';ctx.beginPath();ctx.moveTo(pad,y-22);ctx.lineTo(width-pad,y-22);ctx.stroke();
    ctx.fillStyle='#aee7f6';ctx.font='700 '+Math.round(width*.014)+'px system-ui,sans-serif';ctx.fillText('REOPEN THIS ROI SLICE',pad,y);
    ctx.fillStyle='#e5f4f8';ctx.font=Math.round(width*.013)+'px ui-monospace,SFMono-Regular,monospace';ctx.fillText(roiID(),pad,y+Math.round(width*.023));
    ctx.fillStyle='#a9c1ce';ctx.font=Math.round(width*.011)+'px system-ui,sans-serif';ctx.fillText('axis = acquisition axis · i = zero-based slice · 2.2 µm sampling · not registered to M07',pad,y+Math.round(width*.041));
    ctx.fillText('Lovric et al., PLOS ONE (2017) · doi:10.1371/journal.pone.0183979',pad,y+Math.round(width*.056));
    ctx.fillText('arashabadi.github.io · Created by Arash · github.com/arashabadi/mouse-lung-explorer',pad,y+Math.round(width*.071));
    return out;
  }
  document.getElementById('saveRoiPng').addEventListener('click',()=>{
    if(!alvReady){window.alert('Open the measured-data ROI first.');return;}
    saveCanvasPng(roiCaptureFrame(1920),'mouse-lung-E02-ROI-'+document.getElementById('roiAxis').value+'-'+document.getElementById('roiIndex').value+'.png');
  });
  roiRecordButton.disabled=!roiVideoMime;roiRecordButton.title=roiVideoMime?'Record or stop an E02 ROI video':'Video recording is unavailable in this browser';
  function stopRoiVideo(){if(!roiRecording)return;roiRecording=false;cancelAnimationFrame(roiVideoRaf);roiRecorder.stop();roiRecordButton.textContent='Record ROI video';roiRecordButton.setAttribute('aria-pressed','false')}
  function roiVideoTick(now){if(!roiRecording)return;if(now-roiLastFrame>=1000/30){roiRecorder._captureContext.drawImage(roiCaptureFrame(1280),0,0);roiLastFrame=now}roiVideoRaf=requestAnimationFrame(roiVideoTick)}
  roiRecordButton.addEventListener('click',()=>{
    if(roiRecording){stopRoiVideo();return}
    if(!roiVideoMime||!alvReady)return;
    try{
      const canvas=document.createElement('canvas');canvas.width=1280;canvas.height=720;const context=canvas.getContext('2d');context.drawImage(roiCaptureFrame(1280),0,0);
      roiStream=canvas.captureStream(30);roiRecorder=new MediaRecorder(roiStream,{mimeType:roiVideoMime,videoBitsPerSecond:5000000});roiRecorder._captureContext=context;roiChunks=[];
      roiRecorder.ondataavailable=e=>{if(e.data.size)roiChunks.push(e.data)};
      roiRecorder.onstop=()=>{const type=roiRecorder.mimeType||roiVideoMime,ext=type.startsWith('video/mp4')?'mp4':'webm',blob=new Blob(roiChunks,{type}),url=URL.createObjectURL(blob),a=document.createElement('a');a.href=url;a.download='mouse-lung-E02-ROI-'+document.getElementById('roiAxis').value+'-'+document.getElementById('roiIndex').value+'.'+ext;document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),60000);roiStream.getTracks().forEach(t=>t.stop());roiRecordButton.title='Video ready · '+ext.toUpperCase()};
      roiRecorder.start(1000);roiRecording=true;roiLastFrame=-Infinity;roiRecordButton.textContent='Stop ROI video';roiRecordButton.setAttribute('aria-pressed','true');roiVideoRaf=requestAnimationFrame(roiVideoTick);
    }catch(error){console.error('ROI video failed',error);window.alert('Could not start ROI video: '+error.message)}
  });
"""
old_roi="""  document.getElementById('saveRoiPng').addEventListener('click',()=>{
    if(!alvRenderer){window.alert('Open the measured-data ROI first.');return;}
    alvRenderer.render(alvScene,alvCamera);
    compositeCapture(alvRenderer.domElement,document.getElementById('roiSlice'),'E02 measured-data ROI · exploratory reconstruction','Independent specimen; no registration to M07','mouse-lung-E02-ROI.png');
  });"""
swap(old_roi,roi_video)
swap("document.getElementById('closeAlveolar').addEventListener('click',()=>{alvDialog.hidden=true;});", "document.getElementById('closeAlveolar').addEventListener('click',()=>{if(roiRecording)stopRoiVideo();alvDialog.hidden=true;});")
swap("window.addEventListener('keydown',e=>{if(e.key==='Escape')alvDialog.hidden=true;});", "window.addEventListener('keydown',e=>{if(e.key==='Escape'){if(roiRecording)stopRoiVideo();alvDialog.hidden=true;}});")
swap('<button id="saveRoiPng" type="button">Save ROI PNG</button>', '<button id="saveRoiPng" type="button">Save ROI PNG</button><button id="recordRoiVideo" type="button" aria-pressed="false">Record ROI video</button>')



## Write and validate

The report records the pinned source and generation. Browser checks should include desktop/mobile control layout, Clip opening Section plane, address round-trip, a PNG, and short ROI video.


In [5]:
output=ROOT/'viewer/mouse_lung_v3.1.html'
output.write_text(full)
report={'source_git_commit':BASE_COMMIT,'output':str(output.relative_to(ROOT)),'changes':['collapsed plane panel, expanded view panel, pad-only controls with stepper buttons','compact typeable M07 plane address in export','black section export, short project credit','ROI PNG and video with slice address, website and source DOI'],'bytes':output.stat().st_size}
(ROOT/'qa/v3.1_plane_export_report.json').write_text(json.dumps(report,indent=2)+'\n')
print(f'Built {output} ({output.stat().st_size/2**20:.1f} MiB)')


Built /Users/ashi/github/mouse-lung-explorer/viewer/mouse_lung_v3.1.html (42.8 MiB)
